# Project A - Session 0: compactor precondition

Establishes whether the compactor's keep/drop decision carries any salience signal.
**Nothing downstream is worth running until this passes.**

Pass condition: salience lift beats the positional control printed alongside it, with
zero fallbacks and zero empty keeps.

Expected: only `Qwen2.5-1.5B-Instruct` with `backend: scoring` clears it (2.56x vs a
1.68x control on 6 eval trajectories). Budget ~1 GPU-hour.

In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'   # cleared only if you prefer a Kaggle Dataset
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

QUICK = True
N_EVAL = 4 if QUICK else 24
N_TRAIN = 4 if QUICK else 48
MODELS = ['HuggingFaceTB/SmolLM2-360M-Instruct'] if QUICK else [
    'HuggingFaceTB/SmolLM2-360M-Instruct',
    'Qwen/Qwen2.5-0.5B-Instruct',
    'Qwen/Qwen2.5-1.5B-Instruct',
]
BACKENDS = ['scoring'] if QUICK else ['scoring', 'model']
print(f'QUICK={QUICK}  models={len(MODELS)}  backends={len(BACKENDS)}  n_eval={N_EVAL}')
print(f'{len(MODELS) * len(BACKENDS) * 3} compaction runs queued')


## Datasets

Three sets, because the lift figure differs sharply between them:

- **synthetic (marked)** — facts introduced by `One thing to lock in:`. Debug and
  ratio-sweep set. Measured 2.56x against a 1.68x control.
- **synthetic (unmarked)** — same generator, marker removed. Floor case: facts and
  filler share one template bank and one register. Measured 1.10x against 1.57x.
- **HotpotQA** — natural prose, no marker, supporting sentences scattered through an
  early window. **Primary set.** Measured 1.61x against a 0.98x control at n=60.

The CPU numbers above all used 6 eval trajectories. The point of this session is to
widen them: HotpotQA's margin is only about 2.6 sigma at that size.

In [ ]:
run(f"python data/generate_synthetic.py --n-train {N_TRAIN} --n-eval {N_EVAL} --n-turns 120")
run(f"python data/generate_synthetic.py --n-train {N_TRAIN} --n-eval {N_EVAL} --n-turns 120 --unmarked --out artifacts/data/synthetic_hard")
run(f"python data/load_hotpotqa.py --n-train {N_TRAIN} --n-eval {N_EVAL} --per-trajectory 4")


In [ ]:
run(f"python scripts/preflight.py --config configs/kaggle.yaml")


## Scorer hand-check

Cheap sanity pass before the full sweep: do obviously salient lines outscore obvious
filler? A non-positive separation means the sweep is pointless.

In [ ]:
for m in MODELS:
    run(f"python compactor/inspect_scorer.py --config configs/kaggle.yaml --set model.base={m}")


## Salience lift per compactor

Both elicitation formats, so the index-list failure is reproduced on GPU rather than
taken on trust from the CPU runs.

In [ ]:
SETS = [
    ('synthetic', 'configs/kaggle.yaml', 'artifacts/data/synthetic'),
    ('unmarked', 'configs/kaggle.yaml', 'artifacts/data/synthetic_hard'),
    ('hotpotqa', 'configs/kaggle_hotpotqa.yaml', 'artifacts/data/hotpotqa'),
]
total = len(SETS) * len(BACKENDS) * len(MODELS)
done = 0
for name, cfg, ddir in SETS:
    for backend in BACKENDS:
        for m in MODELS:
            done += 1
            slug = f"{name}_{m.split('/')[-1]}_{backend}"
            out = f'/kaggle/working/artifacts/runs/compactor_check/{slug}'
            print('=' * 70)
            print(f'[{done}/{total}] {slug}', flush=True)
            run(f"python baselines/cascading.py --config {cfg} --split eval --out {out} --set model.base={m} compaction.backend={backend} data.dir={ddir}")
            run(f"python eval/span_report.py --events {out}/eval_events.jsonl --show 0 --out {out}/span_report.json")


In [ ]:
import json, glob
paths = sorted(glob.glob('/kaggle/working/artifacts/runs/compactor_check/*/span_report.json'))
print(f'found {len(paths)} span reports')
if not paths:
    print('NOTHING TO SUMMARISE - the sweep cell did not produce span_report.json files.')
rows = []
for p in paths:
    r = json.load(open(p))
    rows.append((p.split('/')[-2], r['fact_keep_rate'], r['filler_keep_rate'],
                 r['salience_lift'], r['positional_control']['salience_lift'],
                 r['fact_spans']))
print(f"{'config':<44} {'fact':>6} {'filler':>7} {'lift':>7} {'ctrl':>7} {'n':>5}  verdict")
for name, f, fl, lift, ctrl, n in rows:
    verdict = 'USABLE' if lift > ctrl * 1.1 else 'no signal'
    print(f'{name:<44} {f:6.3f} {fl:7.3f} {lift:7.2f} {ctrl:7.2f} {n:5d}  {verdict}')


In [ ]:
run(f"python scripts/kaggle_sync.py save --run-root /kaggle/working/artifacts/runs --archive /kaggle/working/runs.zip")
print('Save /kaggle/working/runs.zip as a Kaggle Dataset before the session ends.')